# 🔀 Sorting Algorithms -- Runnable Notebook

Companion to [`README.md`](README.md).

Insertion sort, merge sort, quicksort -- built from scratch -- then Python's stable `sorted()`, and the
stable multi-key sorting trick.

## 1. Insertion sort -- O(n^2) worst case, O(n) on nearly-sorted input

In [ ]:
def insertion_sort(a):
    a = a[:]                                   # don't mutate the caller's list
    for i in range(1, len(a)):
        key, j = a[i], i - 1
        while j >= 0 and a[j] > key:            # shift larger elements right
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = key
    return a

unsorted = [5, 2, 8, 1, 9, 3]
result = insertion_sort(unsorted)
print("insertion_sort:", result)
assert result == sorted(unsorted)

nearly_sorted = [1, 2, 3, 5, 4, 6, 7]           # only one pair out of place
assert insertion_sort(nearly_sorted) == sorted(nearly_sorted)

## 2. Merge sort -- guaranteed O(n log n), and stable

In [ ]:
def merge(left, right):
    result, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:                 # <= (not <) is what makes this STABLE
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    return result + left[i:] + right[j:]

def merge_sort(a):
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    return merge(merge_sort(a[:mid]), merge_sort(a[mid:]))

unsorted = [5, 3, 8, 1, 9, 2]
result = merge_sort(unsorted)
print("merge_sort:", result)
assert result == sorted(unsorted)

# Stability check: compare ONLY by .value (via __le__), so ties in value are genuine ties
# for the algorithm -- if merge_sort is stable, "first-5" must still precede "second-5".
class Tagged:
    def __init__(self, value, label):
        self.value, self.label = value, label
    def __le__(self, other):
        return self.value <= other.value            # comparison ignores label entirely
    def __repr__(self):
        return f"{self.value}:{self.label}"

tagged = [Tagged(5, "first-5"), Tagged(3, "only-3"), Tagged(5, "second-5"), Tagged(1, "only-1")]
stable_result = merge_sort(tagged)
print("stability check:", stable_result)
fives = [t.label for t in stable_result if t.value == 5]
assert fives == ["first-5", "second-5"]         # original relative order preserved among ties

## 3. Quicksort -- average O(n log n), NOT stable

In [ ]:
def partition(a, lo, hi):
    pivot = a[hi]
    i = lo - 1
    for j in range(lo, hi):
        if a[j] <= pivot:
            i += 1
            a[i], a[j] = a[j], a[i]
    a[i + 1], a[hi] = a[hi], a[i + 1]
    return i + 1

def quicksort(a, lo=0, hi=None):
    if hi is None:
        a = a[:]                                # copy so the caller's list isn't mutated
        hi = len(a) - 1
    if lo < hi:
        p = partition(a, lo, hi)
        quicksort(a, lo, p - 1)
        quicksort(a, p + 1, hi)
    return a

unsorted = [5, 3, 8, 1, 9, 2, 7]
result = quicksort(unsorted)
print("quicksort:", result)
assert result == sorted(unsorted)

already_sorted = list(range(50))                # worst case for a naive "pivot = last element" strategy
assert quicksort(already_sorted) == already_sorted

## 4. Python's `sorted()` -- Timsort, stable, with `key=`

In [ ]:
people = [("bob", 25), ("amy", 30), ("cal", 25)]

by_age = sorted(people, key=lambda p: p[1])
print("by age:", by_age)
assert by_age == [("bob", 25), ("cal", 25), ("amy", 30)]

by_age_desc = sorted(people, key=lambda p: p[1], reverse=True)
print("by age desc:", by_age_desc)
assert by_age_desc == [("amy", 30), ("bob", 25), ("cal", 25)]

# Stability check: two people share age 25 -- their relative order (bob before cal) must survive.
assert by_age.index(("bob", 25)) < by_age.index(("cal", 25))

## 5. Multi-key sort with mixed directions -- exploiting stability

In [ ]:
issues = [
    {"id": "A", "status": "Todo", "assignee": "carol"},
    {"id": "B", "status": "Done", "assignee": "alice"},
    {"id": "C", "status": "Todo", "assignee": "alice"},
]

# Pass 1: sort by the LEAST significant key first (assignee, DESCENDING).
step1 = sorted(issues, key=lambda i: i["assignee"], reverse=True)
print("after pass 1 (assignee desc):", [i["id"] for i in step1])

# Pass 2: sort by the MOST significant key LAST (status, ASCENDING).
# Stability preserves pass 1's ordering wherever status ties.
result = sorted(step1, key=lambda i: i["status"])
result_ids = [i["id"] for i in result]
print("final (status asc, assignee desc within ties):", result_ids)

# B (Done) comes first; within Todo, carol (pass-1 first) stays before alice.
assert result_ids == ["B", "A", "C"]

## ✅ Recap
- Comparison sorts can't beat **O(n log n)** worst case -- `log2(n!)` comparisons are needed to distinguish all orderings.
- **Insertion sort**: O(n^2) worst, O(n) on nearly-sorted data, stable.
- **Merge sort**: guaranteed O(n log n), O(n) space, **stable** (the `<=` in merge is what makes it stable).
- **Quicksort**: O(n log n) average, O(n^2) worst, in-place, **not stable**.
- Python's `sorted()`/`.sort()` use **Timsort** -- O(n) best case, O(n log n) worst, always **stable**.
- Stability lets you sort by multiple keys with **independent directions**: sort least-significant-key first, most-significant-key last.

Next: [`18_Binary_Search`](../18_Binary_Search/README.md).